## Lesson 5: Compare pLMs Across Models and Pooling Strategies
### What you'll learn
- Run a small grid: {models} x {pooling methods} on the same task.
- See how model size and pooling choice affect performance.
- Build a reusable benchmarking pattern you can extend.

### Why we use the "embedding probe" style for comparison
Fine-tuning every model would take hours. We instead use Lesson 1's recipe
(frozen pLM + logistic regression on top) which makes the comparison FAST
and FAIR — no hyperparameter shenanigans, just "which embedding is better?"

If a model wins as a frozen feature extractor, it usually also wins after
fine-tuning. So this is a reasonable cheap proxy.

Output: a CSV table of {model, method, accuracy, F1, time}.

> **Run order matters.** The cells below build on each other. Run them **top to bottom** (Jupyter: *Run → Run All Cells*; VS Code: *Run All*). If you hit `NameError: name 'torch' is not defined` (or similar), you skipped the **Setup** cell — run it first.

## Setup — imports & configuration

**Run this cell first.** It imports every library and defines the module-level constants the rest of the notebook relies on.

In [1]:
import os
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModel, AutoTokenizer
MODELS = [
    # (model name, friendly label)
    ("facebook/esm2_t6_8M_UR50D",   "ESM-2 8M"),
    ("facebook/esm2_t12_35M_UR50D", "ESM-2 35M"),
    # Uncomment if you have a GPU (or significant patience):
    # ("facebook/esm2_t30_150M_UR50D", "ESM-2 150M"),
    # ProtBERT uses a DIFFERENT vocab (spaces between AAs) — see comment in embed():
    # ("Rostlab/prot_bert", "ProtBERT"),
]
METHODS = ["mean-pool", "max-pool", "cls-token"]
DATASET_NAME = "proteinea/solubility"
N_TRAIN = 300
N_TEST = 100
BATCH_SIZE = 8
RESULTS_DIR = "./results"
RESULTS_CSV = os.path.join(RESULTS_DIR, "lesson5_comparison.csv")

### `pool` (function)

Reduce (B, L, D) to (B, D) according to the chosen pooling method.

In [2]:
def pool(hidden, mask, method):
    if method == "mean-pool":
        m = mask.unsqueeze(-1).float()
        return (hidden * m).sum(dim=1) / m.sum(dim=1)
    if method == "max-pool":
        # Set padding positions to -inf so they don't win the max.
        m = mask.unsqueeze(-1).bool()
        masked = hidden.masked_fill(~m, float("-inf"))
        return masked.max(dim=1).values
    if method == "cls-token":
        return hidden[:, 0, :]
    raise ValueError(f"Unknown pooling method: {method}")

### `embed` (function)

Convert a list of sequences into one vector each.

Note on ProtBERT-style models: their tokenizer expects spaces between
amino acids, e.g. "M K T V". If you add a ProtBERT entry above, do
`[' '.join(s) for s in sequences]` before passing in.

In [3]:
def embed(sequences, model, tokenizer, device, method, batch_size=BATCH_SIZE):
    out = []
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            hidden = model(**inputs).last_hidden_state
        pooled = pool(hidden, inputs["attention_mask"], method)
        out.append(pooled.cpu().numpy())
    return np.vstack(out)

### `main` (function)

In [4]:
def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load dataset ONCE — reused across all models.
    print(f"Loading dataset: {DATASET_NAME}")
    ds = load_dataset(DATASET_NAME)
    ds = ds.rename_columns({"sequences": "sequence", "labels": "label"})
    ds = ds.map(lambda b: {"label": [int(x) for x in b["label"]]}, batched=True)
    ds = ds.shuffle(seed=42)
    train = ds["train"].select(range(N_TRAIN))
    test = ds["test"].select(range(N_TEST))
    train_seqs = train["sequence"]
    test_seqs = test["sequence"]
    y_train = np.array(train["label"])
    y_test = np.array(test["label"])

    rows = []
    for model_name, label in MODELS:
        print(f"\n=== {label}  ({model_name}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name).to(device).eval()

        for method in METHODS:
            t0 = time.time()
            X_train = embed(train_seqs, model, tokenizer, device, method)
            X_test = embed(test_seqs, model, tokenizer, device, method)

            clf = LogisticRegression(max_iter=1000)
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)

            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds)
            elapsed = time.time() - t0

            rows.append(
                {
                    "model": label,
                    "method": method,
                    "embed_dim": X_train.shape[1],
                    "accuracy": round(acc, 3),
                    "f1": round(f1, 3),
                    "time_s": round(elapsed, 1),
                }
            )
            print(
                f"  {method:<12} dim={X_train.shape[1]:>4}  acc={acc:.3f}  f1={f1:.3f}  ({elapsed:.0f}s)"
            )

        # Free memory before loading the next model.
        del model, tokenizer
        if device == "cuda":
            torch.cuda.empty_cache()

    # ---- Summary ---------------------------------------------------------
    df = pd.DataFrame(rows)
    print("\n=== Summary ===")
    print(df.to_string(index=False))
    df.to_csv(RESULTS_CSV, index=False)
    print(f"\nSaved comparison table to: {RESULTS_CSV}")

    print(
        """
Things to experiment with:
- Add ProtBERT to MODELS — note the space-between-amino-acids quirk in embed().
- Add a regression task (e.g. proteinea/fluorescence) — swap LogisticRegression
  for sklearn.linear_model.Ridge and use Spearman correlation as the metric.
- Bigger N_TRAIN / N_TEST will give a more reliable comparison.
- Plot the table: import matplotlib.pyplot as plt; df.pivot(...).plot.bar()
"""
    )

## Run the lesson

Execute everything above, then run `main()`.

In [5]:
main()

Using device: cuda
Loading dataset: proteinea/solubility



=== ESM-2 8M  (facebook/esm2_t6_8M_UR50D) ===


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  mean-pool    dim= 320  acc=0.530  f1=0.420  (1s)


...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  mean-pool    dim= 480  acc=0.550  f1=0.416  (2s)


  max-pool     dim= 480  acc=0.590  f1=0.438  (2s)


  cls-token    dim= 480  acc=0.610  f1=0.494  (2s)

=== Summary ===
    model    method  embed_dim  accuracy    f1  time_s
 ESM-2 8M mean-pool        320      0.53 0.420     1.2
 ESM-2 8M  max-pool        320      0.57 0.442     0.9
 ESM-2 8M cls-token        320      0.60 0.487     0.9
ESM-2 35M mean-pool        480      0.55 0.416     1.7
ESM-2 35M  max-pool        480      0.59 0.438     1.8
ESM-2 35M cls-token        480      0.61 0.494     1.7

Saved comparison table to: ./results\lesson5_comparison.csv

Things to experiment with:
- Add ProtBERT to MODELS — note the space-between-amino-acids quirk in embed().
- Add a regression task (e.g. proteinea/fluorescence) — swap LogisticRegression
  for sklearn.linear_model.Ridge and use Spearman correlation as the metric.
- Bigger N_TRAIN / N_TEST will give a more reliable comparison.
- Plot the table: import matplotlib.pyplot as plt; df.pivot(...).plot.bar()

